# Experiment 4 — Calibration Analysis

Evaluates model probability calibration before and after isotonic regression calibration.

**Key outputs:**
- Reliability diagrams (before/after)
- ECE, MCE, Brier Score table
- Sharpness histograms

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.seeding import seed_everything
from src.features.feature_pipeline import CancerTriageFeaturePipeline, load_horizon_data, create_train_val_test_split
from src.models.baselines import BaselineModelTrainer
from src.models.calibration import (
    plot_calibration_curve, calibration_comparison_table, compute_calibration_metrics
)

seed_everything(42)
sns.set_theme(style='whitegrid', font_scale=1.2)
os.makedirs('../results/figures', exist_ok=True)
print('Ready.')

In [ ]:
cfg = {
    'seed': 42,
    'paths': {'data_processed': '../data/processed', 'figures': '../results/figures'},
    'features': {'cbc': ['wbc','hemoglobin','platelets','neutrophils','lymphocytes'],
                 'metabolic': ['glucose','albumin','creatinine'],
                 'inflammatory': ['crp']}
}

X, y = load_horizon_data('../data/processed', horizon_months=12, dataset='mimic')
X_train, X_val, X_test, y_train, y_val, y_test = create_train_val_test_split(X, y)

pipeline = CancerTriageFeaturePipeline(cfg)
X_train_p = pipeline.fit_transform(X_train, y_train)
X_val_p   = pipeline.transform(X_val)
X_test_p  = pipeline.transform(X_test)

drop_cols = ['subject_id','cancer_type','gender','age']
Xte = X_test_p.drop(columns=[c for c in drop_cols if c in X_test_p.columns]).fillna(0)
Xval = X_val_p.drop(columns=[c for c in drop_cols if c in X_val_p.columns]).fillna(0)

trainer = BaselineModelTrainer(cfg)
results = trainer.train_all(X_train_p, y_train, X_val_p, y_val)
best = results['xgboost']['model']

y_prob_uncal = best.predict_proba(Xte)[:, 1]
print('Trained model. Uncalibrated ECE:', compute_calibration_metrics(y_test.to_numpy(), y_prob_uncal)['ece'])

In [ ]:
# Apply isotonic calibration
calibrated = trainer.calibrate_model(best, X_val_p, y_val)
y_prob_cal = calibrated.predict_proba(Xte)[:, 1]

cal_table = calibration_comparison_table({
    'XGBoost': {'y_true': y_test.to_numpy(), 'y_prob_uncal': y_prob_uncal, 'y_prob_cal': y_prob_cal}
})
print(cal_table.to_string())

# Reliability diagram
models_dict = {
    'Uncalibrated XGBoost': (y_test.to_numpy(), y_prob_uncal),
    'Calibrated XGBoost': (y_test.to_numpy(), y_prob_cal)
}
plot_calibration_curve(models_dict, '../results/figures/exp4_calibration.png')
cal_table.to_csv('../results/exp4_calibration_metrics.csv')
print('Saved.')